# European benchmark: GTFS reference-day metrics
**IE Sustainability Datathon (Kearney) · Deliverable 1 · Supporting notebook**

Computes `n_bus_lines` and `daily_bus_km` for each comparison city on a disclosed reference day, filtered to the brief's mandatory network perimeter (section 4.4). Helsinki is the fully reproducible peer. Output: `data_derived/benchmark_gtfs_intermediate.csv`, read by the main notebook, which writes `File_1.csv`. This notebook does not write any mandatory CSV.

In [1]:
# Environment: runs as-is in Google Colab (Python 3.10+).
%pip install -q pandas numpy pyproj shapely folium plotly kaleido requests

import os, io, json, zipfile, datetime as dt, hashlib, textwrap
import numpy as np, pandas as pd, requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

BASE = "/content/datathon" if os.path.exists("/content") else os.path.abspath("..")
RAW, DERIVED, OUT = (os.path.join(BASE, d) for d in ("data_raw", "data_derived", "outputs"))
for d in (RAW, DERIVED, OUT):
    os.makedirs(d, exist_ok=True)

RUN_DATE = dt.date.today().isoformat()
import sys, platform
print(f"Run date {RUN_DATE} | Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}")
print("Folders: data_raw/, data_derived/, outputs/ under", "/content/datathon (Colab)" if BASE.startswith("/content") else "the repo root")

Note: you may need to restart the kernel to use updated packages.


Run date 2026-11-27 | Python 3.13.1 | pandas 2.2.3 | numpy 2.4.6
Folders: data_raw/, data_derived/, outputs/ under the repo root


**Helpers.** `fetch()` downloads each source once and logs URL, date, size and checksum; `assume()` labels every value V, D, P or A for the assumptions register; `check()` prints a visible PASS / FAIL line.

In [2]:
# ---- Data register: every downloaded or manually obtained source is logged here.
DATA_REGISTER = []

def fetch(url, filename, note="", manual=False):
    """Download a public file into data_raw/ and log source, date, size and checksum.
    manual=True: the file was obtained by hand (registration/API); it must already be in data_raw/."""
    path = os.path.join(RAW, filename)
    if not manual and not os.path.exists(path):
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    size = os.path.getsize(path) if os.path.exists(path) else None
    sha = hashlib.sha256(open(path, "rb").read()).hexdigest()[:12] if size else None
    DATA_REGISTER.append(dict(file=filename, url=url, downloaded=RUN_DATE, bytes=size,
                              sha256_12=sha, obtained="manual" if manual else "script", note=note))
    return path

def show_register():
    display(pd.DataFrame(DATA_REGISTER))

# ---- Provenance labels (brief section 4.6): V verified, D derived, P proxy, A assumption.
ASSUMPTIONS = []

def assume(variable, value, label, evidence, impact="", sensitivity=""):
    assert label in {"V", "D", "P", "A"}, "label must be V, D, P or A"
    ASSUMPTIONS.append(dict(variable=variable, value=value, label=label, evidence=evidence,
                            impact=impact, sensitivity=sensitivity))
    return value

# ---- Visible checks: every step prints CHECK ... PASS / FAIL.
CHECKS = []

def check(name, ok, detail=""):
    CHECKS.append(dict(check=name, result="PASS" if ok else "FAIL", detail=detail))
    print(f"CHECK {'PASS' if ok else 'FAIL'} | {name}" + (f" | {detail}" if detail else ""))
    return ok
print("Helpers ready: fetch() logs sources, assume() labels values V/D/P/A, check() prints PASS/FAIL")


Helpers ready: fetch() logs sources, assume() labels values V/D/P/A, check() prints PASS/FAIL


**Fixed parameters (brief §4.5).** Read from notebook 01's parameter cell, so the two notebooks can never disagree on a fixed value.

In [3]:
# The brief's fixed parameters are read out of notebook 01's own parameter cell, never copied here
# (one source of truth for every number in brief section 4.5, including the CO2 factors and their evidence).
def load_P_from_main_notebook():
    path = os.path.join(BASE, "notebooks", "01_Madrid_Electrification_Main.ipynb")
    for c in json.load(open(path))["cells"]:
        s = "".join(c["source"])
        if c["cell_type"] == "code" and "P = dict(" in s:
            g = {"assume": assume, "pd": pd, "np": np, "display": lambda *a, **k: None}
            exec(s, g)
            return g["P"]
    raise RuntimeError("parameter cell 'P = dict(...)' not found in notebook 01")

P = load_P_from_main_notebook()
display(pd.DataFrame([(k, v) for k, v in P.items()], columns=["parameter", "value"]))


,parameter,value
0,kwh_km,"{'Standard': 1.3, 'Articulated': 1.85}"
1,range_km,"{'Standard': 300, 'Articulated': 240}"
2,cng_kg_km,"{'Standard': 0.45, 'Articulated': 0.65}"
3,charger_kw,100
4,window_h,7
5,charge_eff,0.9
6,bus_price_eur,"{'Standard': 615000, 'Articulated': 800000}"
7,benchmark_kwh_km,1.3
8,service_days,365
9,cng_kgco2_per_kg,2.7175


**GTFS helpers.** Read a feed, find the services active on a date (`calendar`, then `calendar_dates` exceptions), and measure trip length from `shape_dist_traveled` or the shape geometry.

In [4]:
# ---- Generic GTFS helpers (shared logic with the benchmark notebook).
def read_gtfs(zip_path, tables=("agency","routes","trips","stop_times","calendar","calendar_dates","shapes","feed_info")):
    feed = {}
    with zipfile.ZipFile(zip_path) as z:
        names = set(z.namelist())
        for t in tables:
            if f"{t}.txt" in names:
                feed[t] = pd.read_csv(z.open(f"{t}.txt"), dtype=str, low_memory=False)
    return feed

def active_service_ids(feed, date_str):
    """Service ids running on date_str (YYYY-MM-DD), applying calendar then calendar_dates exceptions."""
    d = pd.Timestamp(date_str); ymd = d.strftime("%Y%m%d"); dow = d.day_name().lower()
    active = set()
    if "calendar" in feed:
        c = feed["calendar"]
        m = (c["start_date"] <= ymd) & (c["end_date"] >= ymd) & (c[dow] == "1")
        active |= set(c.loc[m, "service_id"])
    if "calendar_dates" in feed:
        cd = feed["calendar_dates"]; cd = cd[cd["date"] == ymd]
        active |= set(cd.loc[cd["exception_type"] == "1", "service_id"])
        active -= set(cd.loc[cd["exception_type"] == "2", "service_id"])
    return active

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*r*np.arcsin(np.sqrt(a))

def shape_lengths_km(feed):
    s = feed["shapes"].copy()
    for c in ("shape_pt_lat","shape_pt_lon"): s[c] = s[c].astype(float)
    s["shape_pt_sequence"] = s["shape_pt_sequence"].astype(int)
    s = s.sort_values(["shape_id","shape_pt_sequence"])
    g = s.groupby("shape_id")
    s["seg"] = haversine_km(g["shape_pt_lat"].shift(), g["shape_pt_lon"].shift(), s["shape_pt_lat"], s["shape_pt_lon"])
    return s.groupby("shape_id")["seg"].sum()

def trip_km(feed, trip_ids, dist_unit_m=None):
    """Km per trip. Preferred: shape_dist_traveled (unit must be confirmed: metres or km);
    fallback: geodesic length of the trip's shape. Prints which method each trip used."""
    st = feed["stop_times"][feed["stop_times"]["trip_id"].isin(trip_ids)]
    km = pd.Series(dtype=float)
    if "shape_dist_traveled" in st and st["shape_dist_traveled"].notna().any():
        d = st.assign(x=pd.to_numeric(st["shape_dist_traveled"], errors="coerce")).groupby("trip_id")["x"].agg(lambda v: v.max() - v.min())
        assert dist_unit_m is not None, "Confirm shape_dist_traveled unit (metres or km) and pass dist_unit_m=True/False"
        km = d / 1000 if dist_unit_m else d
    missing = set(trip_ids) - set(km.dropna().index)
    if missing and "shapes" in feed:
        lens = shape_lengths_km(feed)
        t = feed["trips"].set_index("trip_id").loc[list(missing), "shape_id"]
        km = pd.concat([km.dropna(), t.map(lens)])
    print(f"trip_km: {len(km)} trips, {len(missing)} via shape geometry fallback")
    return km
print("GTFS helpers ready: read_gtfs, active_service_ids, trip-length functions")


GTFS helpers ready: read_gtfs, active_service_ids, trip-length functions


## Cities, feeds and perimeters
The brief requires Madrid plus **at least three** of six eligible capitals: London, Paris, Berlin, Brussels, Helsinki and Copenhagen, each on its mandatory network perimeter (brief §4.4): TfL London Buses · former RATP bus network (Paris and inner ring) · BVG · STIB/MIVB · HSL (whole HSL area) · Movia lines serving Copenhagen and Frederiksberg. At least one city must be fully reproducible from source GTFS (Helsinki). Perimeter filters must be documented per city (operator / agency, `route_type` 3 = bus, exclusions); London's timetables are not one ready-made GTFS, so its extraction route must be documented.

In [5]:
CITIES = {
    # city: dict(feed landing page, operator perimeter, reference date, filter used); every eligible capital is listed
    "Helsinki":   dict(landing="https://www.hsl.fi/en/hsl/open-data", operator="HSL", ref_date="2026-11-24",
                       filter="Every HSL bus route, extended route types 700-799 (HSL uses 701, 702, 704); whole HSL area", in_file1=True),
    "Paris":      dict(landing="https://data.iledefrance-mobilites.fr/explore/dataset/offre-horaires-tc-gtfs-idfm/", operator="RATP (IDFM network)",
                       ref_date="2026-11-24", filter="route_type 3; lines serving the City of Paris, all km (Paris and inner ring); "
                       "km inside Paris printed as a sensitivity", in_file1=True),
    "Copenhagen": dict(landing="https://labs.rejseplanen.dk/hc/da/articles/24750139021341", operator="Movia (agency 221)", ref_date="2026-11-24",
                       filter="route_type 3 and 700; lines serving the Copenhagen + Frederiksberg box, km inside the box", in_file1=True),
    "London":     dict(landing="https://tfl.gov.uk/corporate/publications-and-reports/buses-performance-data", operator="TfL London Buses",
                       ref_date="2026-11-24", filter="No ready GTFS (brief 4.4): TfL annual km / 365 and route count, in notebook 01 §9", in_file1=True),
    "Berlin":     dict(landing="https://unternehmen.vbb.de/en/digital-services/datasets/", operator="BVG", ref_date="2026-11-25",
                       filter="VBB feed, agency_id 796 (BVG) and route_type 700 (bus); every BVG bus line incl. night and rail-replacement lines; Wed 25 Nov 2026 (feed starts that day)", in_file1=True),
    "Brussels":   dict(landing="https://data.belgianmobility.io/en/data.html?agency=stibmivb", operator="STIB-MIVB", ref_date=None,
                       filter="Not covered: the feed found is a March 2026 archive, not processed; five peers already exceed the minimum", in_file1=False),
}
display(pd.DataFrame(CITIES).T)


,landing,operator,ref_date,filter,in_file1
Helsinki,https://www.hsl.fi/en/hsl/open-data,HSL,2026-11-24,"Every HSL bus route, extended route types 700-...",True
Paris,https://data.iledefrance-mobilites.fr/explore/...,RATP (IDFM network),2026-11-24,"route_type 3; lines serving the City of Paris,...",True
Copenhagen,https://labs.rejseplanen.dk/hc/da/articles/247...,Movia (agency 221),2026-11-24,route_type 3 and 700; lines serving the Copenh...,True
London,https://tfl.gov.uk/corporate/publications-and-...,TfL London Buses,2026-11-24,No ready GTFS (brief 4.4): TfL annual km / 365...,True
Berlin,https://unternehmen.vbb.de/en/digital-services...,BVG,2026-11-25,"VBB feed, agency_id 796 (BVG) and route_type 7...",True
Brussels,https://data.belgianmobility.io/en/data.html?a...,STIB-MIVB,None,Not covered: the feed found is a March 2026 ar...,False


**City metrics.** For one city: active services on its reference day, routes in the mandatory perimeter, then `n_bus_lines` and `daily_bus_km`.

In [6]:
def city_metrics(city, zip_path, route_filter, dist_unit_m=None):
    """route_filter(feed) -> set of route_ids in the perimeter. Returns n_bus_lines and daily_bus_km on the city's reference day."""
    feed = read_gtfs(zip_path)
    ref = CITIES[city]["ref_date"]
    services = active_service_ids(feed, ref)
    routes = route_filter(feed)
    trips = feed["trips"][feed["trips"].service_id.isin(services) & feed["trips"].route_id.isin(routes)]
    km = trip_km(feed, set(trips.trip_id), dist_unit_m)
    lines = trips.merge(feed["routes"][["route_id", "route_short_name"]], on="route_id").route_short_name.nunique()
    out = dict(city=city, reference_date=ref, n_bus_lines=int(lines), daily_bus_km=round(float(km.sum()), 1), n_trips=len(trips))
    print(out)
    return out
print("city_metrics() ready: n_bus_lines and daily_bus_km for one city on its reference day")


city_metrics() ready: n_bus_lines and daily_bus_km for one city on its reference day


## Helsinki (fully reproducible peer)
HSL feed. The brief's mandatory perimeter for Helsinki is **the whole HSL area** (§4.4: Helsinki, Espoo, Vantaa, Kauniainen, Kerava, Kirkkonummi, Sipoo, Siuntio and Tuusula), so File 1 uses every HSL bus route. HSL's published fleet figures (600 electric buses, 52 %) cover the same area, so the row is consistent.

- **Common peer day, Tue 24 Nov 2026.** The rolling feeds downloaded on 23 Nov 2026 no longer all cover Madrid's 17 Nov, so every GTFS peer uses the first regular Tuesday inside every feed (brief 4.4 lets each city use its own reference day). The snapshot is pinned by date and checksum in the data register.
- **Bus = HSL's extended route types 700–799.** HSL never uses `route_type` 3: its buses are 701 (regular), 702 (trunk) and 704 (local). Tram (0), metro (1), commuter rail (109), ferry (4) and light rail (900) are excluded.
- **City-only view kept as a sensitivity.** Lines serving the City of Helsinki, counting km between two Helsinki stops (stop code `H` + digit), is printed for the report, never written to File 1. It was the first version of this section (PR #9), modelled on the Paris method.
- **No `frequencies.txt`** in this feed: every trip is an individual run, as in Madrid's feed.
- **No vehicle blocks for buses.** `block_id` is only filled for trams, so peak buses on the road come from trips running at the same minute (a lower bound: it leaves out buses waiting at terminals).

In [7]:
# Helsinki feed. HSL rebuilds it regularly, so fetch() pins the snapshot (date + sha256);
# the numbers below are reproducible only from that exact file.
HSL_GTFS = "https://infopalvelut.storage.hsldev.com/gtfs/hsl.zip"
hel_zip = fetch(HSL_GTFS, "hsl_gtfs.zip",
                "HSL GTFS, whole HSL region, snapshot valid 2026-11-10 to 2027-01-08 — "
                "https://www.hsl.fi/en/hsl/open-data")

HEL_REF = assume("Helsinki reference day", "2026-11-24", "A",
                 "Common peer day: first regular Tuesday inside every peer feed downloaded on 23 Nov 2026 (rolling feeds no "
                 "longer cover Madrid's 2026-11-17); same day as Copenhagen; brief 4.4 lets each city use its own local reference day",
                 "File 1 Helsinki row")
CITIES["Helsinki"]["ref_date"] = HEL_REF      # runtime only: the shared CITIES cell is left untouched

HEL_BUS_TYPES = assume("Helsinki bus route types", "700-799 (HSL uses 701, 702, 704)", "V",
                       "GTFS extended route types; HSL routes.txt has no route_type 3", "File 1 Helsinki row")
HEL_PERIMETER = assume("Helsinki perimeter", "every HSL bus route (whole HSL area)", "V",
                       "Brief section 4.4: HSL (whole HSL area); HSL fleet figures cover the same area",
                       "File 1 Helsinki row", "City-only view printed as a sensitivity below")
HEL_STOP_PREFIX = assume("Helsinki city stops (sensitivity only)", "stop_code starting with H + digit", "P",
                         "HSL stop codes carry the municipality letter (H Helsinki, E Espoo, V Vantaa, ...); "
                         "Hy (Hyvinkaa) is excluded by requiring a digit after H", "City-only sensitivity")
print("feed:", os.path.basename(hel_zip), "| reference day:", HEL_REF)

feed: hsl_gtfs.zip | reference day: 2026-11-24


**Read and clean.** Same steps as the Madrid data preparation (notebook 03): read everything as text, strip padded text, drop exact duplicate rows, then select the bus trips active on the reference day. `stop_times.txt` (1.1 GB) is not loaded here; it is streamed in the next cell.

In [8]:
hel_feed = read_gtfs(hel_zip, tables=("agency", "routes", "trips", "calendar", "calendar_dates", "stops", "shapes", "feed_info"))
_clean = []
for t, df in hel_feed.items():
    obj = df                                   # read_gtfs reads every column as text
    padded = int((obj.notna() & (obj != obj.apply(lambda c: c.str.strip()))).sum().sum())
    df = df.apply(lambda c: c.str.strip()); dups = int(df.duplicated().sum())
    hel_feed[t] = df.drop_duplicates()
    _clean.append(dict(table=t, rows=len(df), padded_cells=padded, exact_duplicates=dups))
display(pd.DataFrame(_clean))

hel_services = active_service_ids(hel_feed, HEL_REF)
hel_bus = hel_feed["routes"][hel_feed["routes"].route_type.str.fullmatch(r"7\d\d")]
ht = hel_feed["trips"]
ht = ht[ht.service_id.isin(hel_services) & ht.route_id.isin(set(hel_bus.route_id))]
_stops = hel_feed["stops"]
hel_city_stops = set(_stops.loc[_stops.stop_code.str.match(r"^H\d", na=False), "stop_id"])

_fi = hel_feed["feed_info"].iloc[0]
check("Helsinki: reference day inside the feed validity",
      _fi.feed_start_date <= HEL_REF.replace("-", "") <= _fi.feed_end_date,
      f"feed valid {_fi.feed_start_date} to {_fi.feed_end_date}")
check("Helsinki: no exact duplicate trips", not ht.duplicated().any() and ht.trip_id.is_unique)
check("Helsinki: every active bus trip has a shape", bool(ht.shape_id.notna().all()))
check("Helsinki: feed has no frequencies.txt (trips are individual runs)",
      "frequencies.txt" not in zipfile.ZipFile(hel_zip).namelist())
print(f"{len(ht)} active bus trips | {ht.route_id.nunique()} bus routes | "
      f"route types {sorted(hel_bus.route_type.unique())} | {len(hel_city_stops)} Helsinki city stops")

,table,rows,padded_cells,exact_duplicates
0,agency,1,0,0
1,routes,460,0,0
2,trips,444330,0,0
3,calendar,8882,0,0
4,calendar_dates,4158,0,0
5,stops,8413,16333,0
6,shapes,497447,0,0
7,feed_info,1,0,0


CHECK PASS | Helsinki: reference day inside the feed validity | feed valid 20261120 to 20270118
CHECK PASS | Helsinki: no exact duplicate trips
CHECK PASS | Helsinki: every active bus trip has a shape
CHECK PASS | Helsinki: feed has no frequencies.txt (trips are individual runs)
19286 active bus trips | 390 bus routes | route types ['701', '702', '704'] | 2653 Helsinki city stops


**Km and times per trip.** This feed has `shape_dist_traveled` in **km** on every stop time, so trip length = last − first value (the main notebook's method A for Madrid). Each trip's first and last departure time give when it is on the road. For the city-only sensitivity, a stop-to-stop segment counts as *inside Helsinki* when both stops are Helsinki stops. `stop_times.txt` (1.1 GB) is streamed in chunks and only the active bus trips are kept. The shape geometry gives an independent cross-check.

In [9]:
_ids, _parts = set(ht.trip_id), []
with zipfile.ZipFile(hel_zip) as z:
    for ch in pd.read_csv(z.open("stop_times.txt"), chunksize=3_000_000, dtype=str,
                          usecols=["trip_id", "stop_id", "stop_sequence", "departure_time", "shape_dist_traveled"]):
        ch = ch[ch.trip_id.isin(_ids)]
        if len(ch):
            _parts.append(ch)
hst = pd.concat(_parts)
hst = hst.assign(seq=hst.stop_sequence.astype(int), d=hst.shape_dist_traveled.astype(float),
                 h=hst.stop_id.str.strip().isin(hel_city_stops),
                 sec=hst.departure_time.str.split(":", expand=True).astype(int).pipe(lambda x: x[0] * 3600 + x[1] * 60 + x[2])
                 ).sort_values(["trip_id", "seq"])
_g = hst.groupby("trip_id")
hst["seg"] = hst.d - _g.d.shift()
_prev_h = _g.h.shift().eq(True)
hst["both_in"] = hst.h & _prev_h
hst["crossing"] = (hst.h ^ _prev_h) & hst.seg.notna()

_per = hst.groupby("trip_id").agg(km=("d", lambda v: v.max() - v.min()), serves_helsinki=("h", "any"),
                                  start_s=("sec", "min"), end_s=("sec", "max"))
_per["km_in_helsinki"] = hst[hst.both_in].groupby("trip_id").seg.sum().reindex(_per.index).fillna(0.0)
_per["km_crossing"] = hst[hst.crossing].groupby("trip_id").seg.sum().reindex(_per.index).fillna(0.0)
_shape_km = hel_feed["shapes"].assign(x=lambda s: s.shape_dist_traveled.astype(float)).groupby("shape_id").x.max()
ht = (ht.merge(_per, left_on="trip_id", right_index=True, how="left")
        .merge(hel_bus[["route_id", "route_short_name", "route_type"]], on="route_id", how="left")
        .assign(shape_km=lambda t: t.shape_id.map(_shape_km)))

check("Helsinki: every active bus trip has stop times", bool(ht.km.notna().all()))
check("Helsinki: every trip has positive km", bool((ht.km > 0).all()))
_ratio = ht.km / ht.shape_km
check("Helsinki: stop-time km within 1% of shape length for >= 99% of trips",
      bool(((_ratio - 1).abs() < 0.01).mean() >= 0.99), f"median ratio {_ratio.median():.4f}")
check("Helsinki: km inside the city never exceed the trip's total km",
      bool((ht.km_in_helsinki <= ht.km + 1e-6).all()))
display(ht.groupby("route_type").agg(trips=("trip_id", "size"), km=("km", "sum")).round(1))

CHECK PASS | Helsinki: every active bus trip has stop times
CHECK PASS | Helsinki: every trip has positive km
CHECK PASS | Helsinki: stop-time km within 1% of shape length for >= 99% of trips | median ratio 1.0000
CHECK PASS | Helsinki: km inside the city never exceed the trip's total km


,trips,km
route_type,,
701,16101,228018.7
702,2780,57386.4
704,405,4397.1


**Perimeter and result.** File 1 uses the whole HSL area (brief §4.4). The other perimeters are printed for the report but never written to `File_1`. The per-line table flags high-km lines with the 1.5 × IQR rule; they are real trunk lines and are **kept**, as in the Madrid data preparation.

In [10]:
_all = pd.Series(True, index=ht.index)
_opts = {
    "Serves Helsinki, km inside Helsinki":          (ht.serves_helsinki, ht.km_in_helsinki),
    "  + half of the boundary-crossing segments":   (ht.serves_helsinki, ht.km_in_helsinki + 0.5 * ht.km_crossing),
    "Serves Helsinki, all km of those lines":       (ht.serves_helsinki, ht.km),
    "Helsinki internal lines only (route_id 1xxx)": (ht.route_id.str.startswith("1"), ht.km),
    "Whole HSL region":                             (_all, ht.km),
}
hel_perims = pd.DataFrame([dict(perimeter=k, n_bus_lines=int(ht.loc[m, "route_short_name"].nunique()),
                                daily_bus_km=round(float(v[m].sum()), 1), n_trips=int(m.sum()))
                           for k, (m, v) in _opts.items()])
display(hel_perims)

_km = dict(zip(hel_perims.perimeter, hel_perims.daily_bus_km))
print(f"City-only sensitivity: {_km['Serves Helsinki, km inside Helsinki']:,.0f} km/day "
      f"({_km['Serves Helsinki, km inside Helsinki'] / _km['Whole HSL region']:.0%} of the HSL area)")

hel_lines = (ht.groupby("route_short_name")
             .agg(trips=("trip_id", "size"), km=("km", "sum")).round(1))
_q1, _q3 = hel_lines.km.quantile([0.25, 0.75])
hel_lines["iqr_outlier"] = hel_lines.km > _q3 + 1.5 * (_q3 - _q1)
print(f"{int(hel_lines.iqr_outlier.sum())} high-km lines flagged (kept):")
display(hel_lines[hel_lines.iqr_outlier].sort_values("km", ascending=False))

helsinki = dict(city="Helsinki", reference_date=HEL_REF,
                n_bus_lines=int(ht.route_short_name.nunique()),
                daily_bus_km=round(float(ht.km.sum()), 1),
                n_trips=int(len(ht)))
check("Helsinki: result matches the city_metrics() contract",
      set(helsinki) == {"city", "reference_date", "n_bus_lines", "daily_bus_km", "n_trips"})
check("Helsinki: File 1 row uses the whole HSL area (brief 4.4)",
      helsinki["daily_bus_km"] == _km["Whole HSL region"])
print(helsinki)

,perimeter,n_bus_lines,daily_bus_km,n_trips
0,"Serves Helsinki, km inside Helsinki",183,131418.3,12372
1,+ half of the boundary-crossing segments,183,136511.3,12372
2,"Serves Helsinki, all km of those lines",183,185963.4,12372
3,Helsinki internal lines only (route_id 1xxx),102,94200.2,8423
4,Whole HSL region,390,289802.2,19286


City-only sensitivity: 131,418 km/day (45% of the HSL area)
30 high-km lines flagged (kept):


,trips,km,iqr_outlier
route_short_name,,,
560,230,6442.5,True
510,221,6305.2,True
520,218,5691.3,True
600,255,5321.3,True
530,204,5091.9,True
570,216,4724.6,True
200,195,4570.0,True
300,208,3717.5,True
400,202,3653.9,True


CHECK PASS | Helsinki: result matches the city_metrics() contract
CHECK PASS | Helsinki: File 1 row uses the whole HSL area (brief 4.4)
{'city': 'Helsinki', 'reference_date': '2026-11-24', 'n_bus_lines': 390, 'daily_bus_km': 289802.2, 'n_trips': 19286}


**Peak buses on the road.** HSL publishes its electric bus count but not its total fleet, so the main notebook derives the total from the electric share (600 ÷ 0.52 ≈ 1,154). The timetable is an independent check: the most bus trips running at the same minute on the reference day. It is a lower bound on buses in service, because buses waiting at a terminal between trips are not counted; the fleet also holds spares.

In [11]:
_grid = np.arange(0, 30 * 3600, 60)
_on_road = np.zeros(len(_grid), dtype=int)
for _s, _e in zip(ht.start_s, ht.end_s):
    _on_road[(_grid >= _s) & (_grid < _e)] += 1
_pk = int(_on_road.argmax())
HEL_PEAK_ON_ROAD = assume("Helsinki peak bus trips running at once", int(_on_road.max()), "D",
                          f"HSL GTFS, {HEL_REF}, at {_grid[_pk] // 3600:02d}:{_grid[_pk] % 3600 // 60:02d}; "
                          "lower bound on buses in service (no layovers, no spares)",
                          "Cross-check of the File 1 fleet_total")
_hours = pd.Series({f"{h:02d}:00": int(_on_road[(_grid >= h * 3600) & (_grid < (h + 1) * 3600)].max())
                    for h in range(5, 25)}, name="max trips running")
display(_hours.to_frame().T)
check("Helsinki: peak trips on the road below the derived fleet of about 1,154",
      HEL_PEAK_ON_ROAD < 1154, f"{HEL_PEAK_ON_ROAD} at peak = {HEL_PEAK_ON_ROAD / 1154:.0%} of 1,154")
print(f"Peak: {HEL_PEAK_ON_ROAD} buses on the road at {_grid[_pk] // 3600:02d}:{_grid[_pk] % 3600 // 60:02d}")

,05:00,06:00,07:00,08:00,09:00,10:00,11:00,12:00,13:00,14:00,15:00,16:00,17:00,18:00,19:00,20:00,21:00,22:00,23:00,24:00
max trips running,427,703,917,934,812,662,629,683,745,870,921,918,852,716,579,510,435,341,290,181


CHECK PASS | Helsinki: peak trips on the road below the derived fleet of about 1,154 | 934 at peak = 81% of 1,154
Peak: 934 buses on the road at 08:08


## Berlin
VBB feed (all Berlin-Brandenburg operators). The brief's mandatory perimeter for Berlin is **the BVG bus network** (§4.4), so File 1 uses every route with `agency_id` 796 (Berliner Verkehrsbetriebe) and `route_type` 700 (bus) that has at least one trip on the reference day, night lines and special services included. BVG tram (900), U-Bahn (400) and ferry (1000) routes are excluded.

- **Reference day deviation, Wed 25 Nov 2026.** The feed downloaded on 26 Nov 2026 starts on 25 Nov 2026, so the peers' common day (Tue 24 Nov) is outside it. 25 Nov is a Wednesday working day: no public holiday in Berlin, outside Berlin's school holidays (autumn 19–31 Oct, Christmas from 23 Dec). Its bus timetable matches Tue 1 Dec and Wed 2 Dec within 0.3 % of trips; both are printed as a sensitivity. The snapshot is pinned by date and checksum in the data register.
- **Km from shape geometry.** Neither `stop_times.txt` nor `shapes.txt` has `shape_dist_traveled`, so trip length is the haversine length of the trip's shape, the same fallback as `trip_km()` and the Copenhagen section. `shapes.txt` (198 MB) is streamed in chunks; `stop_times.txt` (401 MB) is not read.
- **Lines counted by `route_short_name`**, as for Helsinki and Paris. In this feed each BVG line has a single `route_id`, so both counts agree.
- **Rail-replacement buses are kept.** BVG runs some buses under tram or U-Bahn line numbers (for example U6, M8, 12) while those lines are closed for works. They are bus trips in the BVG bus network on the day, so they count; the total without them is printed as a sensitivity.
- **No `frequencies.txt` rows**: every trip is an individual run.

In [12]:
# Berlin feed. VBB republishes it regularly, so fetch() pins the snapshot (date + sha256);
# the numbers below are reproducible only from that exact file.
VBB_GTFS = "https://unternehmen.vbb.de/gtfs"
ber_zip = fetch(VBB_GTFS, "vbb_gtfs.zip",
                "VBB GTFS, all Berlin-Brandenburg operators, downloaded 26 Nov 2026, calendar valid 2026-11-25 to 2027-12-11 — "
                "https://unternehmen.vbb.de/gtfs")
check("Berlin: feed file is a GTFS zip", zipfile.is_zipfile(ber_zip),
      "if FAIL, the URL returned a web page: put the 26 Nov 2026 file in data_raw/vbb_gtfs.zip")
check("Berlin: feed is the 26 Nov 2026 snapshot used for File 1", DATA_REGISTER[-1]["sha256_12"] == "f06fdfa49a89",
      f"sha256 starts {DATA_REGISTER[-1]['sha256_12']}")

BER_REF = assume("Berlin reference day", "2026-11-25", "A",
                 "The VBB feed downloaded on 26 Nov 2026 starts on 2026-11-25, so the peers' Tue 2026-11-24 is outside it; "
                 "2026-11-25 is a Wednesday working day, not a Berlin public holiday, outside Berlin school holidays "
                 "(autumn 19-31 Oct, Christmas from 23 Dec 2026); brief 4.4 lets each city use its own reference day",
                 "File 1 Berlin row", "Tue 2026-12-01 and Wed 2026-12-02 printed below")
CITIES["Berlin"]["ref_date"] = BER_REF      # runtime only, same as the other cities
BER_PERIMETER = assume("Berlin perimeter", "agency_id 796 (Berliner Verkehrsbetriebe), route_type 700", "V",
                       "Brief 4.4: BVG bus network. VBB agency.txt lists BVG as agency 796; its routes are 700 bus, "
                       "900 tram, 400 U-Bahn, 1000 ferry; only 700 is kept", "File 1 Berlin row")
BER_REPLACEMENT = assume("Berlin rail-replacement buses", "included", "A",
                         "Brief 4.4 counts every bus line with a trip on the day, special services included; BVG runs these "
                         "buses under tram or U-Bahn line numbers (route_type 700, agency 796)", "File 1 Berlin row",
                         "Total without them printed below")

ber_feed = read_gtfs(ber_zip, tables=("agency", "routes", "trips", "calendar", "calendar_dates"))
_r = ber_feed["routes"]
ber_bus = _r[(_r.agency_id == "796") & (_r.route_type == "700")]
ber_tram_names = set(_r.loc[(_r.agency_id == "796") & (_r.route_type == "900"), "route_short_name"])
BER_ALT = ["2026-12-01", "2026-12-02"]                  # sensitivity days: next regular Tuesday and Wednesday
ber_days = {}
for _d in [BER_REF] + BER_ALT:
    _s = active_service_ids(ber_feed, _d)
    _t = ber_feed["trips"]
    ber_days[_d] = (_t[_t.service_id.isin(_s) & _t.route_id.isin(set(ber_bus.route_id))]
                    .merge(ber_bus[["route_id", "route_short_name"]], on="route_id", how="left"))
bt = ber_days[BER_REF]

_cal = ber_feed["calendar"]
check("Berlin: agency 796 is Berliner Verkehrsbetriebe",
      ber_feed["agency"].set_index("agency_id").at["796", "agency_name"] == "Berliner Verkehrsbetriebe")
check("Berlin: reference day inside the feed validity",
      _cal.start_date.min() <= BER_REF.replace("-", "") <= _cal.end_date.max(),
      f"feed valid {_cal.start_date.min()} to {_cal.end_date.max()}")
check("Berlin: reference day is a Tuesday or Wednesday (brief 4.4)",
      pd.Timestamp(BER_REF).day_name() in ("Tuesday", "Wednesday"), pd.Timestamp(BER_REF).day_name())
check("Berlin: no duplicate trip ids", bt.trip_id.is_unique)
check("Berlin: every active bus trip has a shape", bool(bt.shape_id.notna().all()),
      f"{int(bt.shape_id.isna().sum())} of {len(bt)} trips without a shape")
with zipfile.ZipFile(ber_zip) as _z:
    check("Berlin: feed has no frequency-based trips (trips are individual runs)",
          "frequencies.txt" not in _z.namelist() or len(pd.read_csv(_z.open("frequencies.txt"))) == 0)
_n = {d: len(t) for d, t in ber_days.items()}
check("Berlin: reference-day bus trips within 1 % of Tue 1 Dec and Wed 2 Dec (normal working day)",
      all(abs(_n[BER_REF] / _n[d] - 1) < 0.01 for d in BER_ALT), ", ".join(f"{d}: {n}" for d, n in _n.items()))
print(f"BVG bus routes in the feed: {len(ber_bus)} | active on {BER_REF}: {bt.route_id.nunique()} routes, "
      f"{len(bt)} trips, {bt.service_id.nunique()} service_ids")

CHECK PASS | Berlin: feed file is a GTFS zip | if FAIL, the URL returned a web page: put the 26 Nov 2026 file in data_raw/vbb_gtfs.zip
CHECK PASS | Berlin: feed is the 26 Nov 2026 snapshot used for File 1 | sha256 starts f06fdfa49a89


CHECK PASS | Berlin: agency 796 is Berliner Verkehrsbetriebe
CHECK PASS | Berlin: reference day inside the feed validity | feed valid 20261125 to 20271211
CHECK PASS | Berlin: reference day is a Tuesday or Wednesday (brief 4.4) | Wednesday
CHECK PASS | Berlin: no duplicate trip ids
CHECK PASS | Berlin: every active bus trip has a shape | 0 of 29207 trips without a shape
CHECK PASS | Berlin: feed has no frequency-based trips (trips are individual runs)
CHECK PASS | Berlin: reference-day bus trips within 1 % of Tue 1 Dec and Wed 2 Dec (normal working day) | 2026-11-25: 29207, 2026-12-01: 29288, 2026-12-02: 29284
BVG bus routes in the feed: 222 | active on 2026-11-25: 215 routes, 29207 trips, 38 service_ids


In [13]:
# Km per trip = haversine length of its shape (no shape_dist_traveled in this feed; same fallback as trip_km()).
# shapes.txt is 198 MB, so it is streamed and only the shapes used by active BVG bus trips are kept.
def stream_shape_km(zip_path, shape_ids, chunk=2_000_000):
    parts = []
    with zipfile.ZipFile(zip_path) as z:
        for ch in pd.read_csv(z.open("shapes.txt"), chunksize=chunk, dtype={"shape_id": str},
                              usecols=["shape_id", "shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"]):
            ch = ch[ch.shape_id.isin(shape_ids)]
            if len(ch):
                parts.append(ch)
    s = pd.concat(parts).sort_values(["shape_id", "shape_pt_sequence"])
    g = s.groupby("shape_id")
    s["seg"] = haversine_km(g.shape_pt_lat.shift(), g.shape_pt_lon.shift(), s.shape_pt_lat, s.shape_pt_lon)
    return s.groupby("shape_id").seg.sum(), int(s.duplicated(["shape_id", "shape_pt_sequence"]).sum())

ber_shape_km, _dups = stream_shape_km(ber_zip, set().union(*[set(t.shape_id.dropna()) for t in ber_days.values()]))
for _d in ber_days:
    ber_days[_d] = ber_days[_d].assign(km=ber_days[_d].shape_id.map(ber_shape_km))
bt = ber_days[BER_REF]
bt["night"] = bt.route_short_name.str.startswith("N")
bt["replacement"] = bt.route_short_name.isin(ber_tram_names) | bt.route_short_name.str.startswith("U")

check("Berlin: every active bus trip has a shape length", bool(bt.km.notna().all()),
      f"{int(bt.km.isna().sum())} of {len(bt)} trips without a shape length")
check("Berlin: every trip has positive km", bool((bt.km > 0).all()))
check("Berlin: no repeated point sequence inside a shape", _dups == 0, f"{_dups} repeated (shape_id, sequence) pairs")
print(f"{len(ber_shape_km)} shapes measured | mean trip {bt.km.mean():.2f} km")

CHECK PASS | Berlin: every active bus trip has a shape length | 0 of 29207 trips without a shape length
CHECK PASS | Berlin: every trip has positive km
CHECK PASS | Berlin: no repeated point sequence inside a shape | 0 repeated (shape_id, sequence) pairs
1327 shapes measured | mean trip 9.99 km


In [14]:
# File 1 takes every BVG bus line on the reference day; the other rows are printed for the report only.
_opts = {
    "All BVG bus lines, Wed 25 Nov (File 1)":            (bt, pd.Series(True, index=bt.index)),
    "  without rail-replacement buses":                   (bt, ~bt.replacement),
    "  without night lines (N...)":                       (bt, ~bt.night),
    "All BVG bus lines, Tue 1 Dec":                       (ber_days["2026-12-01"], pd.Series(True, index=ber_days["2026-12-01"].index)),
    "All BVG bus lines, Wed 2 Dec":                       (ber_days["2026-12-02"], pd.Series(True, index=ber_days["2026-12-02"].index)),
}
ber_perims = pd.DataFrame([dict(perimeter=k, n_bus_lines=int(t.loc[m, "route_short_name"].nunique()),
                                daily_bus_km=round(float(t.loc[m, "km"].sum()), 1), n_trips=int(m.sum()))
                           for k, (t, m) in _opts.items()])
display(ber_perims)
print("Rail-replacement lines on the day:", sorted(bt.loc[bt.replacement, "route_short_name"].unique()))

ber_lines = bt.groupby("route_short_name").agg(trips=("trip_id", "size"), km=("km", "sum")).round(1)
_q1, _q3 = ber_lines.km.quantile([0.25, 0.75])
ber_lines["iqr_outlier"] = ber_lines.km > _q3 + 1.5 * (_q3 - _q1)
print(f"{int(ber_lines.iqr_outlier.sum())} high-km line(s) flagged (kept, real trunk lines); threshold {_q3 + 1.5 * (_q3 - _q1):,.0f} km:")
display(ber_lines[ber_lines.iqr_outlier].sort_values("km", ascending=False))
print("Top five lines by km:")
display(ber_lines.sort_values("km", ascending=False).head(5))

berlin = dict(city="Berlin", reference_date=BER_REF,
              n_bus_lines=int(bt.route_short_name.nunique()),
              daily_bus_km=round(float(bt.km.sum()), 1),
              n_trips=int(len(bt)))
_avg = 89.7e6 / 365                                    # BVG in Zahlen 2025: 89.7 M Nutzwagenkilometer, all days
check("Berlin: result matches the city_metrics() contract",
      set(berlin) == {"city", "reference_date", "n_bus_lines", "daily_bus_km", "n_trips"})
check("Berlin: line count and daily km are positive", berlin["n_bus_lines"] > 0 and berlin["daily_bus_km"] > 0)
check("Berlin: working-day km between 1.0 and 1.3 x BVG's all-days average (89.7 M km / 365)",
      1.0 <= berlin["daily_bus_km"] / _avg <= 1.3, f"{berlin['daily_bus_km'] / _avg:.2f} x {_avg:,.0f} km")
print(f"Lines: {berlin['n_bus_lines']} = {int((~bt.night & ~bt.replacement).groupby(bt.route_short_name).any().sum())} day "
      f"+ {bt.loc[bt.night, 'route_short_name'].nunique()} night + {bt.loc[bt.replacement, 'route_short_name'].nunique()} "
      "rail-replacement (BVG, 31 Dec 2025: 155 day and 68 night lines)")
print(berlin)

,perimeter,n_bus_lines,daily_bus_km,n_trips
0,"All BVG bus lines, Wed 25 Nov (File 1)",215,291780.3,29207
1,without rail-replacement buses,205,278205.3,26036
2,without night lines (N...),167,281893.7,28367
3,"All BVG bus lines, Tue 1 Dec",215,291542.4,29288
4,"All BVG bus lines, Wed 2 Dec",215,291889.8,29284


Rail-replacement lines on the day: ['12', '21', '27', 'M1', 'M10', 'M17', 'M2', 'M8', 'U2', 'U6']
1 high-km line(s) flagged (kept, real trunk lines); threshold 5,000 km:


,trips,km,iqr_outlier
route_short_name,,,
M11,244,5139.6,True


Top five lines by km:


,trips,km,iqr_outlier
route_short_name,,,
M11,244,5139.6,True
M29,358,4852.7,False
X83,326,4455.5,False
M41,411,4130.9,False
M45,292,4109.8,False


CHECK PASS | Berlin: result matches the city_metrics() contract
CHECK PASS | Berlin: line count and daily km are positive
CHECK PASS | Berlin: working-day km between 1.0 and 1.3 x BVG's all-days average (89.7 M km / 365) | 1.19 x 245,753 km
Lines: 215 = 157 day + 48 night + 10 rail-replacement (BVG, 31 Dec 2025: 155 day and 68 night lines)
{'city': 'Berlin', 'reference_date': '2026-11-25', 'n_bus_lines': 215, 'daily_bus_km': 291780.3, 'n_trips': 29207}


## Paris
IDFM feed (all Île-de-France operators). Two deviations from the original plan, both documented in the cells below:

- The feed is a **rolling 30-day window rebuilt daily**. The copy downloaded on 23 Nov 2026 no longer covers Madrid's 17 Nov, so Paris uses the common peer day, **Tue 24 Nov 2026**; the exact snapshot is pinned by date and checksum in the data register.
- The 2025 bus re-tendering split the former RATP network into geographic sub-networks, so `agency_name == "RATP"` now covers only 2 bus routes. The perimeter is therefore **geographic** (lines serving the City of Paris), not by operator.

`stop_times.txt` (644 MB) is never read: the feed has no `shape_dist_traveled`, so trip length comes from shape geometry, streamed in chunks to stay inside Colab's memory.

In [15]:
# Paris feed. Rolling 30-day window rebuilt daily, so fetch() pins the snapshot (date + sha256);
# the numbers below are reproducible only from that exact file.
PARIS_GTFS = "https://eu.ftp.opendatasoft.com/stif/GTFS/IDFM-gtfs.zip"
paris_zip = fetch(PARIS_GTFS, "idfm_gtfs.zip",
                  "IDFM GTFS, all Ile-de-France operators, rolling 30-day feed rebuilt daily — "
                  "https://data.iledefrance-mobilites.fr/explore/dataset/offre-horaires-tc-gtfs-idfm/")

PARIS_REF = assume("Paris reference day", "2026-11-24", "A",
                   "Common peer day: first regular Tuesday inside every peer feed downloaded on 23 Nov 2026 (the rolling "
                   "30-day window no longer covers Madrid's 2026-11-17); same day as Helsinki and Copenhagen",
                   "File 1 Paris row")
CITIES["Paris"]["ref_date"] = PARIS_REF      # runtime only: the shared CITIES cell is left untouched

PARIS_BOX = assume("Paris perimeter box", (48.8156, 48.9022, 2.2241, 2.4699), "P",
                   "Bounding box of the City of Paris inside the Boulevard Peripherique; "
                   "a commune polygon would be sharper", "File 1 Paris row")
print("feed:", os.path.basename(paris_zip), "| reference day:", PARIS_REF)

feed: idfm_gtfs.zip | reference day: 2026-11-24


**Shape lengths.** `shapes.txt` is streamed in chunks; each shape gets its total km and its km inside the Paris box.

In [16]:
def paris_shape_km(zip_path, shape_ids, box, chunk=1_000_000):
    """Per shape: total km, km inside `box`, and whether it enters the box at all.
    shapes.txt is 127 MB in this feed, so it is streamed rather than loaded whole."""
    lat0, lat1, lon0, lon1 = box
    tot, inside, touches, carry = {}, {}, {}, {}
    with zipfile.ZipFile(zip_path) as z:
        for ch in pd.read_csv(z.open("shapes.txt"), chunksize=chunk, dtype={"shape_id": str},
                              usecols=["shape_id", "shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"]):
            ch = ch[ch.shape_id.isin(shape_ids)]
            if ch.empty:
                continue
            for sid, g in ch.sort_values(["shape_id", "shape_pt_sequence"]).groupby("shape_id", sort=False):
                la, lo = g.shape_pt_lat.to_numpy(float), g.shape_pt_lon.to_numpy(float)
                if sid in carry:                       # stitch across the chunk boundary
                    la, lo = np.r_[carry[sid][0], la], np.r_[carry[sid][1], lo]
                carry[sid] = (la[-1], lo[-1])
                if len(la) < 2:
                    continue
                seg = haversine_km(la[:-1], lo[:-1], la[1:], lo[1:])
                ins = (la >= lat0) & (la <= lat1) & (lo >= lon0) & (lo <= lon1)
                tot[sid] = tot.get(sid, 0.0) + float(seg.sum())
                inside[sid] = inside.get(sid, 0.0) + float(seg[ins[:-1] & ins[1:]].sum())
                touches[sid] = touches.get(sid, False) or bool(ins.any())
    return tot, inside, touches
print("paris_shape_km() ready: total km and km inside the Paris box per shape, streamed in chunks")


paris_shape_km() ready: total km and km inside the Paris box per shape, streamed in chunks


**Active bus trips on the reference day.** Trips on bus routes running that day, with total km, km inside Paris and whether they enter Paris.

In [17]:
# Active bus trips on the reference day. stop_times is not in the table list: it is 644 MB and
# unnecessary here, since this feed has no shape_dist_traveled.
paris_feed = read_gtfs(paris_zip, tables=("agency", "routes", "trips", "calendar", "calendar_dates"))
paris_services = active_service_ids(paris_feed, PARIS_REF)
paris_bus = (paris_feed["routes"][paris_feed["routes"].route_type == "3"]
             .merge(paris_feed["agency"][["agency_id", "agency_name"]], on="agency_id", how="left"))

pt = paris_feed["trips"]
pt = pt[pt.service_id.isin(paris_services) & pt.route_id.isin(set(paris_bus.route_id)) & pt.shape_id.notna()]
_tot, _ins, _touch = paris_shape_km(paris_zip, set(pt.shape_id), PARIS_BOX)
pt = (pt.assign(km=pt.shape_id.map(_tot),
                km_in_paris=pt.shape_id.map(_ins),
                serves_paris=pt.shape_id.map(_touch).fillna(False).astype(bool))
        .merge(paris_bus[["route_id", "route_short_name", "agency_name"]], on="route_id", how="left"))

_cal = paris_feed["calendar"]
check("Paris: reference day inside the feed validity",
      _cal.start_date.min() <= PARIS_REF.replace("-", "") <= _cal.end_date.max(),
      f"feed valid {_cal.start_date.min()} to {_cal.end_date.max()}")
check("Paris: every active bus trip has a shape length", bool(pt.km.notna().all()),
      f"{int(pt.km.isna().sum())} of {len(pt)} trips without a shape")
check("Paris: km inside the box never exceed the trip's total km",
      bool((pt.km_in_paris <= pt.km + 1e-6).all()))
print(f"{len(pt)} active bus trips | {pt.route_short_name.nunique()} lines | "
      f"{pt.agency_name.nunique()} networks in the region")

CHECK PASS | Paris: reference day inside the feed validity | feed valid 20261120 to 20261222
CHECK PASS | Paris: every active bus trip has a shape length | 0 of 118258 trips without a shape
CHECK PASS | Paris: km inside the box never exceed the trip's total km
118258 active bus trips | 1854 lines | 63 networks in the region


**Candidate perimeters.** What each perimeter gives; File 1 takes lines serving Paris with all their km, and km inside Paris is the sensitivity.

In [18]:
# Supplementary (printed only, never written to File_1): what each candidate perimeter would give.
_opts = {
    "Serves Paris, km inside Paris":       (pt.serves_paris, "km_in_paris"),
    "Serves Paris, all km of those lines": (pt.serves_paris, "km"),
    "Rive Droite + Rive Gauche only":      (pt.agency_name.isin(["Rive Droite", "Rive Gauche"]), "km"),
    "Whole Ile-de-France region":          (pd.Series(True, index=pt.index), "km"),
}
paris_perims = pd.DataFrame([dict(perimeter=k,
                                  n_bus_lines=int(pt.loc[m, "route_short_name"].nunique()),
                                  daily_bus_km=round(float(pt.loc[m, c].sum()), 1),
                                  n_trips=int(m.sum()))
                             for k, (m, c) in _opts.items()])
display(paris_perims)

_alt = float(paris_perims.loc[paris_perims.perimeter == "Serves Paris, km inside Paris", "daily_bus_km"].iloc[0])
PARIS_PERIMETER = assume("Paris perimeter", "bus lines serving the City of Paris, all km of those lines", "P",
                         "Brief 4.4: former RATP network, Paris and inner ring. No operator field identifies it after the 2025 "
                         "re-tendering, so lines serving Paris are taken with their full length into the inner ring; this matches "
                         "RATP's fleet figure (4,700 buses, Paris and inner ring) better than km inside Paris only",
                         "File 1 Paris row",
                         f"{_alt:,.0f} km/day if only km inside the City of Paris are counted")

_m, _c = _opts["Serves Paris, all km of those lines"]
paris = dict(city="Paris", reference_date=PARIS_REF,
             n_bus_lines=int(pt.loc[_m, "route_short_name"].nunique()),
             daily_bus_km=round(float(pt.loc[_m, _c].sum()), 1),
             n_trips=int(_m.sum()))
check("Paris: result matches the city_metrics() contract",
      set(paris) == {"city", "reference_date", "n_bus_lines", "daily_bus_km", "n_trips"})
check("Paris: line count and daily km are positive",
      paris["n_bus_lines"] > 0 and paris["daily_bus_km"] > 0)
print(f"File 1 uses: lines serving Paris, all km of those lines = {paris['daily_bus_km']:,.1f} km/day on {paris['n_bus_lines']} lines; "
      f"sensitivity: km inside the City of Paris only = {_alt:,.1f} km/day")
print(paris)

,perimeter,n_bus_lines,daily_bus_km,n_trips
0,"Serves Paris, km inside Paris",286,212137.8,38778
1,"Serves Paris, all km of those lines",286,421322.4,38778
2,Rive Droite + Rive Gauche only,66,116076.2,12575
3,Whole Ile-de-France region,1854,1354325.5,118258


CHECK PASS | Paris: result matches the city_metrics() contract
CHECK PASS | Paris: line count and daily km are positive
File 1 uses: lines serving Paris, all km of those lines = 421,322.4 km/day on 286 lines; sensitivity: km inside the City of Paris only = 212,137.8 km/day
{'city': 'Paris', 'reference_date': '2026-11-24', 'n_bus_lines': 286, 'daily_bus_km': 421322.4, 'n_trips': 38778}


## Brussels (not covered)
STIB-MIVB is eligible, but the only feed found is a March 2026 archive, which was not processed; File 1 already has five peers (Helsinki, Paris, London, Copenhagen, Berlin), more than the brief's minimum of three.

In [19]:
brussels = None
print("Brussels: not covered (March 2026 archive feed not processed; File 1 has five peers)")


Brussels: not covered (March 2026 archive feed not processed; File 1 has five peers)


## Copenhagen
Rejseplanen's national GTFS feed (Denmark, Sweden, Germany, Austria) carries Movia (`agency_id` 221) alongside every other Danish/Nordic operator. Perimeter per brief §4.4: **Movia bus lines serving Copenhagen and Frederiksberg**. The feed has no municipality or zone field, so the two municipalities are approximated by a lat/lon box (`CPH_BOX`, below) rather than their true polygon boundary — labelled **P** (proxy), same caveat as Helsinki/Paris's own city-perimeter choices.

- **Reference day deviation.** This snapshot's `calendar` is only valid from **2026-11-20**, so Madrid's reference day (2026-11-17, Tuesday, §2) falls just outside it. Uses the nearest Tuesday inside the window, **2026-11-24**, same logic as the open item on `peak_buses`'s reference day.
- **Bus route types.** Movia uses `route_type` **3** (local bus) and **700** (S-bus, Copenhagen's high-frequency trunk routes, e.g. 1A, 2A, 5C, 150S–600S). `route_type` 4 (ferry) is excluded.
- Rejseplanen rebuilds this feed roughly every two weeks, so (like Paris) it is pinned by date and checksum in the data register; the numbers below are reproducible only from that exact file.
- **In File 1.** Lines and km below feed File 1's Copenhagen row; notebook 01 §9 adds fleet, zero-emission fleet, target, charging model and Denmark's grid factor (Energinet Miljødeklaration 2025).
- **Km basis differs from Paris.** Copenhagen counts only the km run *inside* the box (Movia lines run far into the region), while Paris counts the *full length* of every line serving the City of Paris (the brief's former-RATP perimeter). Copenhagen's km is therefore the tighter, city-only measure; both alternatives are printed below.
- **Fleet proxy.** Movia publishes one fleet for its whole region (1,126 buses), so notebook 01 scales it by this perimeter's share of all Movia bus km on the same day and feed (`movia_all_bus_km`, computed at the end of this section) — label **P**.

In [20]:
# Rejseplanen's national feed. Movia is agency_id 221 inside it; the feed itself has no
# shape_dist_traveled, so km comes from shape geometry only (same fallback trip_km() uses).
REJSEPLANEN_GTFS = "https://www.rejseplanen.info/labs/GTFS.zip"
cph_zip = fetch(REJSEPLANEN_GTFS, "rejseplanen_gtfs.zip",
                "Rejseplanen national GTFS (DK/SE/DE/AT), snapshot valid 2026-11-20 to 2027-02-10 — "
                "https://labs.rejseplanen.dk/hc/da/articles/24750139021341")

CPH_REF = assume("Copenhagen reference day", "2026-11-24", "A",
                  "Nearest Tuesday to Madrid's 2026-11-17 reference day inside this snapshot's validity "
                  "window (2026-11-20 to 2027-02-10); same approach as the peak_buses reference-day open item")

cph_feed = read_gtfs(cph_zip, tables=("agency", "routes", "trips", "calendar", "calendar_dates", "stops", "shapes"))
movia_bus_routes = set(cph_feed["routes"].loc[
    (cph_feed["routes"].agency_id == "221") & (cph_feed["routes"].route_type.isin(["3", "700"])), "route_id"])
cph_services = active_service_ids(cph_feed, CPH_REF)
cph_cand = cph_feed["trips"][cph_feed["trips"].route_id.isin(movia_bus_routes) & cph_feed["trips"].service_id.isin(cph_services)]
print(f"Movia bus routes: {len(movia_bus_routes)} | candidate trips on {CPH_REF}: {len(cph_cand)}")

Movia bus routes: 374 | candidate trips on 2026-11-24: 18623


**Copenhagen + Frederiksberg box.** No zone/municipality field in this feed, so the perimeter is a lat/lon box (proxy); trips are kept if any stop they visit falls inside it, checked by streaming `stop_times` in chunks (the file is ~230MB).

In [21]:
# Copenhagen + Frederiksberg, approximated as a bounding box (brief §4.4 names the two
# municipalities; the feed has no polygon or zone field to filter on directly).
CPH_BOX = assume("Copenhagen+Frederiksberg box", dict(lat_min=55.61, lat_max=55.73, lon_min=12.44, lon_max=12.65), "P",
                  "Approximate bounding box around the two municipalities (no authoritative polygon in this feed); "
                  "same style of proxy as Helsinki/Paris's own city-perimeter choice")

cph_stops = cph_feed["stops"].assign(lat=cph_feed["stops"].stop_lat.astype(float), lon=cph_feed["stops"].stop_lon.astype(float))
box_stop_ids = set(cph_stops.loc[
    cph_stops.lat.between(CPH_BOX["lat_min"], CPH_BOX["lat_max"]) & cph_stops.lon.between(CPH_BOX["lon_min"], CPH_BOX["lon_max"]),
    "stop_id"])
check("Copenhagen+Frederiksberg box contains a plausible stop count", 500 < len(box_stop_ids) < 5000, f"{len(box_stop_ids)} stops")

# stop_times for this feed is ~230MB / 4.2M rows, so it is streamed in chunks and only trip_id/stop_id
# are kept, same approach as Helsinki/Paris for their large files.
cand_ids = set(cph_cand.trip_id)
serves_box = {}
for ch in pd.read_csv(zipfile.ZipFile(cph_zip).open("stop_times.txt"), chunksize=1_000_000, dtype=str, usecols=["trip_id", "stop_id"]):
    ch = ch[ch.trip_id.isin(cand_ids)]
    if len(ch):
        g = ch.assign(inbox=ch.stop_id.isin(box_stop_ids)).groupby("trip_id")["inbox"].any()
        for tid, v in g.items():
            serves_box[tid] = serves_box.get(tid, False) or bool(v)
serving_trip_ids = {t for t, v in serves_box.items() if v}
print(f"candidate trips: {len(cand_ids)} | trips serving the box: {len(serving_trip_ids)}")

CHECK PASS | Copenhagen+Frederiksberg box contains a plausible stop count | 1670 stops


candidate trips: 18623 | trips serving the box: 7807


**Km per trip.** No `shape_dist_traveled` in this feed, so length comes from shape geometry (haversine), split into “inside the box” vs “crossing” segments — same both-ends-inside rule as Helsinki/Paris. All three candidate perimeters are printed; only the “km inside the box” row is used below.

In [22]:
# Per-shape length, split into "inside the box" vs "crossing" segments (both-ends-inside rule,
# same as Helsinki/Paris); this feed has no shape_dist_traveled, so geometry is the only method.
cph_shapes = cph_feed["shapes"].copy()
for c in ("shape_pt_lat", "shape_pt_lon"):
    cph_shapes[c] = cph_shapes[c].astype(float)
cph_shapes["shape_pt_sequence"] = cph_shapes.shape_pt_sequence.astype(int)
cph_shapes = cph_shapes.sort_values(["shape_id", "shape_pt_sequence"])
g = cph_shapes.groupby("shape_id")
cph_shapes["seg"] = haversine_km(g["shape_pt_lat"].shift(), g["shape_pt_lon"].shift(), cph_shapes["shape_pt_lat"], cph_shapes["shape_pt_lon"])
cph_shapes["inbox"] = cph_shapes.shape_pt_lat.between(CPH_BOX["lat_min"], CPH_BOX["lat_max"]) & cph_shapes.shape_pt_lon.between(CPH_BOX["lon_min"], CPH_BOX["lon_max"])
both_in = cph_shapes["inbox"] & g["inbox"].shift().fillna(False)
cph_shapes["seg_in"] = np.where(both_in, cph_shapes["seg"], 0.0)

shape_km = cph_shapes.groupby("shape_id")["seg"].sum()
shape_km_in = cph_shapes.groupby("shape_id")["seg_in"].sum()

ct = (cph_cand[cph_cand.trip_id.isin(serving_trip_ids)]
      .merge(cph_feed["routes"][["route_id", "route_short_name"]], on="route_id"))
ct = ct.assign(km=ct.shape_id.map(shape_km), km_in=ct.shape_id.map(shape_km_in))
ct["km_crossing"] = (ct.km - ct.km_in).clip(lower=0)

_opts = {
    "Serves box, km inside box only":                 (slice(None), ct.km_in),
    "  + half of the boundary-crossing segments":     (slice(None), ct.km_in + 0.5 * ct.km_crossing),
    "Serves box, all km of those lines":               (slice(None), ct.km),
}
cph_perims = pd.DataFrame([dict(perimeter=k, n_bus_lines=int(ct.route_short_name.nunique()),
                                 daily_bus_km=round(float(v.sum()), 1), n_trips=len(ct))
                           for k, (m, v) in _opts.items()])
display(cph_perims)  # printed for the report; never written to File_1 except the chosen row below

_km = dict(zip(cph_perims.perimeter, cph_perims.daily_bus_km))
CPH_PERIMETER = assume("Copenhagen perimeter metric", "Movia bus lines serving the box, km run inside the box", "P",
                        "Same rule as Helsinki/Paris: the agency is regional, so the city network is defined "
                        "geographically; closest analogue to EMT's municipal network", "File 1 Copenhagen row",
                        f"{_km['  + half of the boundary-crossing segments']:,.0f} km/day with half the boundary "
                        f"segments; {_km['Serves box, all km of those lines']:,.0f} km/day with all km of the same lines")

/var/folders/q1/cv3dnhdx7wdg1fdh6vsh0xzh0000gn/T/ipykernel_49273/2867666657.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  both_in = cph_shapes["inbox"] & g["inbox"].shift().fillna(False)


,perimeter,n_bus_lines,daily_bus_km,n_trips
0,"Serves box, km inside box only",57,81334.9,7807
1,+ half of the boundary-crossing segments,57,98958.9,7807
2,"Serves box, all km of those lines",57,116582.9,7807


**Outliers and result.** High-km lines are flagged (1.5×IQR) and kept — they are real trunk routes (5C is Denmark's busiest bus line). Final row matches the `city_metrics()` contract. The last lines compute all Movia bus km on the same day, the denominator of notebook 01's fleet proxy.

In [23]:
cph_lines = ct.groupby("route_short_name").agg(trips=("trip_id", "size"), km_in=("km_in", "sum"), km_total=("km", "sum")).round(1)
_q1, _q3 = cph_lines.km_in.quantile([0.25, 0.75])
cph_lines["iqr_outlier"] = cph_lines.km_in > _q3 + 1.5 * (_q3 - _q1)
print(f"{int(cph_lines.iqr_outlier.sum())} high-km lines flagged (kept, real trunk routes):")
display(cph_lines[cph_lines.iqr_outlier].sort_values("km_in", ascending=False))
_zero = cph_lines[cph_lines.km_in == 0]
print(f"{len(_zero)} line(s) touch the box but run 0 km between two inside stops: {list(_zero.index)} "
      "(counted as serving the box, same rule as Helsinki/Paris)")

copenhagen = dict(city="Copenhagen", reference_date=CPH_REF,
                   n_bus_lines=int(ct.route_short_name.nunique()),
                   daily_bus_km=round(float(ct.km_in.sum()), 1),
                   n_trips=len(ct))
check("Copenhagen: result matches the city_metrics() contract",
      set(copenhagen) == {"city", "reference_date", "n_bus_lines", "daily_bus_km", "n_trips"})
check("Copenhagen: line count and daily km are positive",
      copenhagen["n_bus_lines"] > 0 and copenhagen["daily_bus_km"] > 0)
print(copenhagen)

# All Movia bus km on the reference day (every Movia bus trip, whole region, same feed and shape geometry):
# denominator of notebook 01's fleet proxy, Movia fleet x (perimeter km / all Movia bus km).
_all_km = cph_cand.shape_id.map(shape_km)
check("Copenhagen: every Movia bus trip on the reference day has a shape", _all_km.notna().all(),
      f"{int(_all_km.isna().sum())} trips without shape")
movia_all_bus_km = round(float(_all_km.sum()), 1)
check("Copenhagen: perimeter km is a share of all Movia bus km", 0 < copenhagen["daily_bus_km"] < movia_all_bus_km,
      f"{copenhagen['daily_bus_km']:,.1f} of {movia_all_bus_km:,.1f} km = {copenhagen['daily_bus_km'] / movia_all_bus_km:.1%}")
print(f"All Movia bus trips on {CPH_REF}: {len(cph_cand)} trips, "
      f"{cph_feed['routes'].set_index('route_id').loc[sorted(set(cph_cand.route_id)), 'route_short_name'].nunique()} lines, "
      f"{movia_all_bus_km:,.1f} km")


2 high-km lines flagged (kept, real trunk routes):


,trips,km_in,km_total,iqr_outlier
route_short_name,,,,
5C,491,8164.9,8418.8,True
1A,339,6533.1,6744.0,True


0 line(s) touch the box but run 0 km between two inside stops: [] (counted as serving the box, same rule as Helsinki/Paris)
CHECK PASS | Copenhagen: result matches the city_metrics() contract
CHECK PASS | Copenhagen: line count and daily km are positive
{'city': 'Copenhagen', 'reference_date': '2026-11-24', 'n_bus_lines': 57, 'daily_bus_km': 81334.9, 'n_trips': 7807}
CHECK PASS | Copenhagen: every Movia bus trip on the reference day has a shape | 0 trips without shape
CHECK PASS | Copenhagen: perimeter km is a share of all Movia bus km | 81,334.9 of 333,915.1 km = 24.4%
All Movia bus trips on 2026-11-24: 18623 trips, 359 lines, 333,915.1 km


## Output
One row per city; fields not computable from GTFS are left for the main notebook (fleet, targets, charging model, grid factor). `operator_daily_bus_km` (Copenhagen only) is all Movia bus km on the same day, used by notebook 01 for the fleet proxy.

In [24]:
results = [r for r in (globals().get(k) for k in ("helsinki", "berlin", "paris", "brussels", "copenhagen")) if isinstance(r, dict)]
out = pd.DataFrame(results)
out["operator_daily_bus_km"] = out.city.map({"Copenhagen": movia_all_bus_km})   # fleet-proxy denominator
out.to_csv(os.path.join(DERIVED, "benchmark_gtfs_intermediate.csv"), index=False, encoding="utf-8")
display(out)
check("Benchmark intermediate written with at least one GTFS-computed peer", len(out) >= 1, ", ".join(out.city))
show_register()

,city,reference_date,n_bus_lines,daily_bus_km,n_trips,operator_daily_bus_km
0,Helsinki,2026-11-24,390,289802.2,19286,NaN
1,Berlin,2026-11-25,215,291780.3,29207,NaN
2,Paris,2026-11-24,286,421322.4,38778,NaN
3,Copenhagen,2026-11-24,57,81334.9,7807,333915.1


CHECK PASS | Benchmark intermediate written with at least one GTFS-computed peer | Helsinki, Berlin, Paris, Copenhagen


,file,url,downloaded,bytes,sha256_12,obtained,note
0,hsl_gtfs.zip,https://infopalvelut.storage.hsldev.com/gtfs/h...,2026-11-27,91096014,70f93697e54f,script,"HSL GTFS, whole HSL region, snapshot valid 202..."
1,vbb_gtfs.zip,https://unternehmen.vbb.de/gtfs,2026-11-27,84067147,f06fdfa49a89,script,"VBB GTFS, all Berlin-Brandenburg operators, do..."
2,idfm_gtfs.zip,https://eu.ftp.opendatasoft.com/stif/GTFS/IDFM...,2026-11-27,127072175,4ee32000da2a,script,"IDFM GTFS, all Ile-de-France operators, rollin..."
3,rejseplanen_gtfs.zip,https://www.rejseplanen.info/labs/GTFS.zip,2026-11-27,53922869,228b4923e285,script,"Rejseplanen national GTFS (DK/SE/DE/AT), snaps..."
